# 10b | Machine Learning: The Full scikit-learn Workflow

This is the standard task in take-home assignments and ML-style OAs: you get a dataset, build a predictive model, and explain your approach. Grading is mostly not about how high the score is but whether **the process is correct**: no leakage, sensible splitting and evaluation, a baseline, and honest conclusions.

This lesson walks through the whole process on the app's large built-in dataset (8,000 users). About 70 minutes.

**Task:** predict whether a user converts (`converted`).

In [ ]:
import numpy as np
import pandas as pd
from practice_data import load_capstone

users, events = load_capstone()
print(users.shape, events.shape)
print(users.isna().sum())
print("Conversion rate:", users["converted"].mean().round(3))
users.head()

## 1. Step one is always checking for leakage

For every column, ask: **at the moment we need to make the prediction, does this value already exist?**

Look at how `revenue_30d` relates to the target:

In [ ]:
print(users.groupby("converted")["revenue_30d"].agg(["mean", "min", "max"]))

### ✏️ Exercise 1 (write it yourself)

The columns of `users` are: `user_id`, `country`, `age`, `acq_channel`, `signup_date`, `treatment`, `converted`, `revenue_30d`.

Put the names of the columns that **should not** be used as features into a list `bad_cols`. (Don't include the target `converted`; ignore `signup_date` for now and don't include it either.)

In [ ]:
bad_cols = []
print(bad_cols)

<details>
<summary><b>Hint</b></summary>

One column is a "consequence of the outcome" (target leakage); the other is just an identifier.

</details>

<details>
<summary><b>Answer</b></summary>

```python
bad_cols = ["user_id", "revenue_30d"]
print(bad_cols)
```

`revenue_30d` is the amount spent after conversion. Put it in the model and the AUC will be 1.0 — it looks perfect and is useless in production. Leakage in real data is usually less obvious, so whenever results look too good to be true, suspect leakage first.

</details>

## 2. Feature engineering: aggregating features from an event log

`events` records each user's actions (`view`, `click`, `search`, `purchase`). Aggregate it into features with "one row per user," then merge them onto `users`. This uses the `groupby` / `pivot_table` / `merge` from 02c and 02d.

The `purchase` event is itself the conversion, so it **cannot** be used.

### ✏️ Exercise 2 (write it yourself)

Build a table `feats` with one row per user, containing `user_id` and three columns `view`, `click`, `search`: the number of events of each type for that user.

Then left-join it onto `users` to get `data`; users with no events of a given type get a count of 0. `data` should have 8,000 rows and contain all columns of `users` plus these three.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

Filter out `purchase` first. Then `pivot_table(index="user_id", columns="event_type", values="event_date", aggfunc="count", fill_value=0).reset_index()`. Then `users.merge(feats, on="user_id", how="left")`, and finally `fillna(0)` on the three columns.

</details>

<details>
<summary><b>Answer</b></summary>

```python
feats = (
    events[events["event_type"] != "purchase"]
    .pivot_table(index="user_id", columns="event_type", values="event_date", aggfunc="count", fill_value=0)
    .reset_index()
)
data = users.merge(feats, on="user_id", how="left")
data[["view", "click", "search"]] = data[["view", "click", "search"]].fillna(0)
print(data.shape)
data.head()
```

</details>

> **There is actually a hidden temporal-leakage risk here.** If these actions happened **after** the user converted, they are also "consequences of the outcome." In a real project you would set a prediction time and build features only from actions **before** that time. This practice dataset doesn't make that distinction, but in an interview or take-home you should raise the issue yourself — it earns a lot of credit.

## 3. Splitting

- `X` is the feature table and `y` the target;
- `stratify=y` makes the training and test sets have the same conversion rate;
- **Once the test set is split off, set it aside and use it only once, at the very end.**

### ✏️ Exercise 3 (fill in the blanks)

Hold out 20% as a test set, with stratified sampling and `random_state=0`.

In [ ]:
from sklearn.model_selection import train_test_split

num_cols = ["age", "treatment", "view", "click", "search"]
cat_cols = ["country", "acq_channel"]
X = data[num_cols + cat_cols]
y = data["converted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=____, stratify=____, random_state=0)
print(X_train.shape, X_test.shape, y_train.mean().round(4), y_test.mean().round(4))

<details>
<summary><b>Answer</b></summary>

```python
from sklearn.model_selection import train_test_split

num_cols = ["age", "treatment", "view", "click", "search"]
cat_cols = ["country", "acq_channel"]
X = data[num_cols + cat_cols]
y = data["converted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print(X_train.shape, X_test.shape, y_train.mean().round(4), y_test.mean().round(4))
```

</details>

## 4. Preprocessing: ColumnTransformer

Different column types need different handling:

- Numeric columns: impute missing values with the median (`age` has 250 missing) → standardize;
- Categorical columns: one-hot encode. `handle_unknown="ignore"` means a category not seen in training won't raise an error at prediction time.

`ColumnTransformer` bundles "what to do to which columns," and it goes into a `Pipeline` together with the model. That way, during cross-validation each fold computes the medians, means, standard deviations, and category lists from its training part only, with no leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

def make_pipeline(model):
    preprocess = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("pre", preprocess), ("model", model)])

baseline = make_pipeline(LogisticRegression(max_iter=1000))
baseline.fit(X_train, y_train)
print(baseline.predict_proba(X_train.head(3))[:, 1].round(3))

## 5. Cross-validation

`cross_val_score(model, X, y, cv=5, scoring="roc_auc")` returns 5 validation scores. **Do this on the training set only.**

### ✏️ Exercise 4 (write it yourself)

Run 5-fold cross-validation of `baseline` on the training set (`scoring="roc_auc"`) and store the mean of the 5 scores in `cv_auc`.

In [ ]:
from sklearn.model_selection import cross_val_score

# Write your code below

<details>
<summary><b>Hint</b></summary>

`scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring="roc_auc")`, then take `.mean()`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring="roc_auc")
cv_auc = scores.mean()
print(scores.round(4), round(cv_auc, 4))
```

</details>

## 6. Comparing several models

All sklearn models share the same interface, so switching models is a one-line change. Below we compare logistic regression, a random forest, and gradient boosting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

candidates = {
    "logistic": LogisticRegression(max_iter=1000),
    "random forest": RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=0),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
for name, model in candidates.items():
    s = cross_val_score(make_pipeline(model), X_train, y_train, cv=5, scoring="roc_auc")
    print(f"{name:18s} AUC = {s.mean():.4f} ± {s.std():.4f}")

**🗣 Say it out loud:** In the results above, logistic regression is no worse than the tree models — even slightly better. How do you explain this? How should you write it up in a take-home?

<details>
<summary><b>Reference answer</b></summary>

This dataset has few features, and the signal is roughly linear (monotone) with few complex interactions to learn; a more flexible model then only adds variance and, with default parameters, overfits slightly. This is common: model complexity should match the signal in the data.

**How to write it up:** report it honestly. "The cross-validated AUCs of the three models are …; logistic regression is best, and the differences are about one standard deviation. Since it is simpler, interpretable, and easy to calibrate, I chose it as the final model." This shows more judgment than forcing a complex model. You can also point out that adding the behavioral features earlier improved the score far more than changing models — feature engineering mattered more than model choice.

**Short version:** The signal here is mostly linear with few interactions, so flexible models add variance without reducing bias. I'd report that honestly and choose the simpler, interpretable model, noting that feature engineering mattered far more than model choice.

</details>

## 7. Tuning hyperparameters: GridSearchCV

`GridSearchCV` cross-validates every combination of hyperparameters and picks the one with the highest mean score.

Parameter names inside a Pipeline are written `stepname__paramname` (two underscores). Our model step is called `"model"`, so logistic regression's `C` is written `"model__C"`.

After fitting: `gs.best_params_` is the best parameter set, `gs.best_score_` is its cross-validation score, and `gs.best_estimator_` is the model refit on the full training set with the best parameters.

### ✏️ Exercise 5 (fill in the blanks)

Search over five candidate values of `C` and store the best `C` in `best_C`.

In [ ]:
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(
    make_pipeline(LogisticRegression(max_iter=1000)),
    param_grid={"____": [0.001, 0.01, 0.1, 1, 10]},
    cv=5,
    scoring="roc_auc",
)
gs.fit(X_train, ____)
best_C = gs.best_params_["model__C"]
print(best_C, round(gs.best_score_, 4))

<details>
<summary><b>Answer</b></summary>

```python
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(
    make_pipeline(LogisticRegression(max_iter=1000)),
    param_grid={"model__C": [0.001, 0.01, 0.1, 1, 10]},
    cv=5,
    scoring="roc_auc",
)
gs.fit(X_train, y_train)
best_C = gs.best_params_["model__C"]
print(best_C, round(gs.best_score_, 4))
```

The scores of the five candidates differ very little. That is a finding too: this model is insensitive to the regularization strength. With many parameters, `RandomizedSearchCV` (random sampling) is more efficient than an exhaustive grid.

</details>

## 8. Finally: evaluate once on the test set

This is the first time we touch the test set. Report this number, and then **do not change the model based on it**.

### ✏️ Exercise 6 (write it yourself)

Use `gs.best_estimator_` to predict conversion probabilities on the test set, compute the test ROC-AUC, and store it in `test_auc`.

In [ ]:
from sklearn.metrics import roc_auc_score

# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
from sklearn.metrics import roc_auc_score

proba = gs.best_estimator_.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, proba)
print(round(test_auc, 4))
```

</details>

## 9. What did the model learn?

Pull the individual steps out of the Pipeline: `pipe.named_steps["pre"]`, `pipe.named_steps["model"]`. Use `get_feature_names_out()` for the feature names after preprocessing.

In [ ]:
best = gs.best_estimator_
names = best.named_steps["pre"].get_feature_names_out()
coefs = best.named_steps["model"].coef_[0]
print(pd.Series(coefs, index=names).sort_values(key=abs, ascending=False).round(3))

The numeric features are standardized, so the absolute sizes of the coefficients are directly comparable: the behavior counts (especially `view`) are the strongest predictors.

## Take-home checklist

Before submitting, confirm each item:

1. **Problem definition:** what you predict, at what point in time, with which metric, and why;
2. **Data checks:** row count, missingness, duplicates, distribution of the target;
3. **Leakage:** whether each feature is available at prediction time;
4. **Splitting:** stratified / by time / grouped by user, and why;
5. **Baseline:** how a simple model (even "always guess the majority class") performs;
6. **Preprocessing inside a Pipeline**, fit on training data only;
7. **Cross-validation** to choose the model and hyperparameters;
8. **Use the test set only once**;
9. **Interpretation:** which features matter, and whether the results make sense;
10. **Limitations and next steps:** what you didn't do and what you would do with more time.

Item 10 is often skipped, but graders value it highly.

Next: **11a**.